# HAR-RV and ML Models for Volatility forecasting

### HAR-RV = Heterogeneous AutoRegressive model for Realized Volatility

Financial markets have participants operating on very different time horizons simultaneously — a high-frequency market maker cares about volatility over the next hour, a hedge fund over the next week, a pension fund over the next month. Each group's trading behavior generates volatility at their own scale, and these scales interact.
Standard GARCH models have one "memory" — the single α+β persistence parameter. They can't separately capture short-term clustering and long-term mean reversion at the same time. A GARCH model fit to daily data essentially averages these different components into one number and loses information.

It models future realized variance/volatility measured over multiple time scales:

$$ RV_{t+1} = \beta_0 + \beta_d RV^d_{t} + \beta_w \overline{RV}^w_{t-4:t} + \beta_m \overline{RV}^m_{t-21:t} + \epsilon_{t+1} $$

where
* $ RV^d_{t-1}$ is yesterday's realized variance (1-day)

* $ \overline{RV}^w_{t-1}$ is average of the pass week's(5 days) realized variance

* $ \overline{RV}^m_{t-1}$ is the average of the past month's (22 days) realized variance

It is essentially an OLS regression with three features. It works because it matches the empirical structure of volatility.

**Volatility has long memory:** If you plot the autocorrelation of daily realized variance, it decays extremely slowly-- there is still meaningful autocorrelation at lags of 50, 100, even 200 days.

GARCH(1,1) has short memory by construction; it's autocorrelation decays geometrically fast. HAR approximated long memory by stacking components at three horizons, which empirically captures the slow decay well without requiring a complex model.

HAR is linear. This means it is fast, interpretable, and you can derive prediction intervals. The coefficients are economically meaningful -- typically $ \beta_d > \beta_w > \beta_m > 0$, meaning recent volatility matters most but all three horizons contribute. It usually beats GARCH out-of-sample. The reason is that the 5-day and 22-day components act as a kind of regularization — they smooth out noise in the 1-day component and give the model a better estimate of the "volatility regime" you're currently in.

In [2]:
import sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

sys.path.append(r"C:\Users\arbaz2\Desktop\Quant Finance\Volatility Forecasting\src")

from data_pipeline import make_dataset
from baselines import make_baseline_forecasts
from metrics import qlike, mse, score, score_by_regime, score_by_ticker
from garch import add_garch_refit_recurse, add_gjr_garch_forecast, plot_vol_compare

CRISIS_WINDOWS = {
    "GFC_2007_2009": ("2007-07-01", "2009-06-30"),
    "COVID_2020": ("2020-02-15", "2020-05-31"),
}



In [3]:
df = make_dataset(["SPY", "JPM"], start="2000-01-01", horizons=(1,5), crisis_windows=CRISIS_WINDOWS)
df["date"] = pd.to_datetime(df["date"])

## Plotting the AutoCorrelation of daily realized volatility to show that it has long memory

* AutoCorrelation Function (ACF) at lag 1 should be strong
* ACF at lag 5-7 (a week) should still be meaningful
* ACF at lag 21 (a month) and even 100 is often nontrivial

In [3]:
import plotly.graph_objects as go

def plot_rv_acf(df, ticker, max_lag=150):
    """
    Plot autocorrelation of daily realized variance rv1_var
    for one ticker.
    """
    d = df[df["ticker"] == ticker].sort_values("date").copy()
    x = d["rv1_var"].dropna()

    acf_vals = [x.autocorr(lag=lag) for lag in range(1, max_lag + 1)]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=list(range(1, max_lag + 1)),
        y=acf_vals,
        name="ACF"
    ))

    # highlight important lags
    for lag in [1, 7, 21, 100]:
        if lag <= max_lag:
            fig.add_vline(x=lag, line_dash="dash", line_color="red")

    fig.update_layout(
        title=f"{ticker}: Autocorrelation of Daily Realized Variance",
        xaxis_title="Lag (days)",
        yaxis_title="Autocorrelation"
    )

    fig.show()

In [4]:
plot_rv_acf(df, "SPY", max_lag=150)
plot_rv_acf(df, "JPM", max_lag=150)

In [4]:
df_f = make_baseline_forecasts(df, hv_window=20, ewma_lam=0.94)
df_f.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.605412,4723500,-1.733110,3.003669,5.704724,14.393675,calm,NaN,NaN,5.354321,26.771603
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,92.128937,5741700,0.342534,0.117329,1.449362,36.093206,calm,NaN,NaN,1.480348,7.401742
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,22.071888,8405550,-2.388456,5.704724,0.390285,14.400518,calm,NaN,NaN,5.213282,26.066408
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,91.026451,7503700,-1.203895,1.449362,0.999549,37.060789,calm,NaN,NaN,1.398567,6.992836
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,22.210209,7271850,0.624728,0.390285,2.253408,14.666621,calm,NaN,NaN,5.242768,26.213841


In [5]:
df_garch = add_garch_refit_recurse(df_f, refit_every=5, min_train=750, mean="zero", dist="t")
df_garch.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var,garch1_var,garch5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.605412,4723500,-1.733110,3.003669,5.704724,14.393675,calm,NaN,NaN,5.354321,26.771603,NaN,NaN
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,92.128937,5741700,0.342534,0.117329,1.449362,36.093206,calm,NaN,NaN,1.480348,7.401742,NaN,NaN
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,22.071888,8405550,-2.388456,5.704724,0.390285,14.400518,calm,NaN,NaN,5.213282,26.066408,NaN,NaN
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,91.026451,7503700,-1.203895,1.449362,0.999549,37.060789,calm,NaN,NaN,1.398567,6.992836,NaN,NaN
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,22.210209,7271850,0.624728,0.390285,2.253408,14.666621,calm,NaN,NaN,5.242768,26.213841,NaN,NaN


In [7]:
df_gjr_garch = add_gjr_garch_forecast(df_garch, refit_every=21, min_train=750, dist="t" )
df_gjr_garch.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,...,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var,garch1_var,garch5_var,gjr1_var,gjr5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.605412,4723500,-1.733110,3.003669,...,14.393675,calm,NaN,NaN,5.354321,26.771603,NaN,NaN,NaN,NaN
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,92.128937,5741700,0.342534,0.117329,...,36.093206,calm,NaN,NaN,1.480348,7.401742,NaN,NaN,NaN,NaN
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,22.071888,8405550,-2.388456,5.704724,...,14.400518,calm,NaN,NaN,5.213282,26.066408,NaN,NaN,NaN,NaN
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,91.026451,7503700,-1.203895,1.449362,...,37.060789,calm,NaN,NaN,1.398567,6.992836,NaN,NaN,NaN,NaN
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,22.210209,7271850,0.624728,0.390285,...,14.666621,calm,NaN,NaN,5.242768,26.213841,NaN,NaN,NaN,NaN


In [8]:
models_1d = ["hv1_var","ewma1_var","garch1_var","gjr1_var"]
print(score(df_gjr_garch, models_1d, "rv1_var", eval_start="2005-01-01"))
print("\n")

models_5d = ["hv5_var", "ewma5_var", "garch5_var","gjr5_var"]
print(score(df_gjr_garch, models_5d, "rv5_var", eval_start="2005-01-01"))
print("\n")
print(score_by_regime(df_gjr_garch, models_1d, "rv1_var", eval_start="2005-01-01").head(30))

        model   target      n     QLIKE         MSE
3    gjr1_var  rv1_var  10652  1.359028  213.055874
2  garch1_var  rv1_var  10652  1.393296  220.020207
1   ewma1_var  rv1_var  10652  1.450791  226.174898
0     hv1_var  rv1_var  10652  1.490452  235.490591


        model   target      n     QLIKE          MSE
3    gjr5_var  rv5_var  10652  2.831691   999.880640
2  garch5_var  rv5_var  10652  2.844596  1058.152795
0     hv5_var  rv5_var  10652  2.890881  1456.071512
1   ewma5_var  rv5_var  10652  2.911794  1330.494781


           regime       model   target     n     QLIKE          MSE
3      COVID_2020    gjr1_var  rv1_var   144  3.724725  1546.460710
2      COVID_2020  garch1_var  rv1_var   144  3.968351  1576.126226
0      COVID_2020     hv1_var  rv1_var   144  4.259607  1823.975506
1      COVID_2020   ewma1_var  rv1_var   144  4.591994  1750.479876
7   GFC_2007_2009    gjr1_var  rv1_var  1008  2.948959  1834.599204
6   GFC_2007_2009  garch1_var  rv1_var  1008  3.004557  1900.09

In [9]:
plot_vol_compare(df_gjr_garch, "SPY", target_var="rv1_var", forecast_vars=("hv1_var","ewma1_var","garch1_var", "gjr1_var"), crisis_windows = CRISIS_WINDOWS )

In [10]:
def make_har_features_single(d):
    """
    Create HAR-RV features for a single ticker.

    Uses lagged realized variance at 3 time scales:
    - rv_d : yesterday's realized variance
    - rv_w : average realized variance over last 5 days
    - rv_m : average realized variance over last 22 days
    """
    d = d.sort_values("date").copy()

    # 1-day lag
    d["rv_d"] = d["rv1_var"].shift(1)

    # 5-day average lag
    d["rv_w"] = d["rv1_var"].shift(1).rolling(window=5).mean()

    # 22-day average lag
    d["rv_m"] = d["rv1_var"].shift(1).rolling(window=22).mean()

    return d

In [11]:
def add_har_features(df):
    """
    Apply HAR feature construction separately to each ticker.
    """
    out = df.copy().sort_values(["ticker", "date"]).reset_index(drop=True)

    pieces = []
    for tkr in out["ticker"].unique():
        d = out[out["ticker"] == tkr].copy()
        d = make_har_features_single(d)
        pieces.append(d)

    out = pd.concat(pieces, axis=0).sort_values(["date", "ticker"]).reset_index(drop=True)
    return out

In [12]:
def har_rolling_forecast_single(
    d,
    target_col="rv1_var",
    refit_every=21,
    min_train=252
):
    """
    Expanding-window HAR-RV forecast for one ticker.

    - Fits linear regression on lagged HAR features
    - Refits every `refit_every` days
    - Forecasts 1-day realized variance
    """
    d = d.sort_values("date").copy()

    feature_cols = ["rv_d", "rv_w", "rv_m"]
    fcast = np.full(len(d), np.nan)

    t = min_train

    while t < len(d):
        # training sample up to time t
        train = d.iloc[:t+1].dropna(subset=feature_cols + [target_col]).copy()

        if len(train) < min_train:
            t += refit_every
            continue

        # fit linear regression on HAR features
        X_train = train[feature_cols].values
        y_train = train[target_col].values

        model = LinearRegression()
        model.fit(X_train, y_train)

        # forecast until next refit
        t_end = min(t + refit_every, len(d))

        for i in range(t, t_end):
            row = d.iloc[i]

            if row[feature_cols].isna().any():
                continue

            X_test = row[feature_cols].values.reshape(1, -1)
            fcast[i] = model.predict(X_test)[0]

        t = t_end

    return pd.Series(fcast, index=d.index)

In [13]:
def add_har_forecasts(
    df,
    refit_every=21,
    min_train=252
):
    """
    Add HAR-RV forecasts to the panel dataframe.

    Adds:
    - har1_var : 1-day forecast
    - har5_var : simple 5-day approximation = 5 * har1_var
    """
    out = add_har_features(df)
    out = out.sort_values(["ticker", "date"]).reset_index(drop=True)

    forecasts = []

    for tkr in out["ticker"].unique():
        d = out[out["ticker"] == tkr].copy()

        s = har_rolling_forecast_single(
            d,
            target_col="rv1_var",
            refit_every=refit_every,
            min_train=min_train
        )

        forecasts.append(s.rename(tkr))

    har_all = pd.concat(forecasts, axis=0).sort_index()

    out["har1_var"] = har_all
    out["har5_var"] = 5.0 * out["har1_var"]

    return out.sort_values(["date", "ticker"]).reset_index(drop=True)

In [14]:
df_har = add_har_forecasts(df_gjr_garch, refit_every=21, min_train=252)

In [15]:
models_1d = ["hv1_var", "ewma1_var", "garch1_var", "gjr1_var", "har1_var"]
print(score(df_har, models_1d, "rv1_var", eval_start="2005-01-01"))
print()

models_5d = ["hv5_var", "ewma5_var", "garch5_var", "gjr5_var", "har5_var"]
print(score(df_har, models_5d, "rv5_var", eval_start="2005-01-01"))
print()

print(score_by_regime(df_har, models_1d, "rv1_var", eval_start="2005-01-01"))

        model   target      n     QLIKE         MSE
3    gjr1_var  rv1_var  10652  1.359028  213.055874
2  garch1_var  rv1_var  10652  1.393296  220.020207
4    har1_var  rv1_var  10652  1.446939  221.666354
1   ewma1_var  rv1_var  10652  1.450791  226.174898
0     hv1_var  rv1_var  10652  1.490452  235.490591

        model   target      n     QLIKE          MSE
3    gjr5_var  rv5_var  10652  2.831691   999.880640
2  garch5_var  rv5_var  10652  2.844596  1058.152795
0     hv5_var  rv5_var  10652  2.890881  1456.071512
1   ewma5_var  rv5_var  10652  2.911794  1330.494781
4    har5_var  rv5_var  10652  2.936052   821.363259

           regime       model   target     n     QLIKE          MSE
3      COVID_2020    gjr1_var  rv1_var   144  3.724725  1546.460710
4      COVID_2020    har1_var  rv1_var   144  3.935383  1460.542384
2      COVID_2020  garch1_var  rv1_var   144  3.968351  1576.126226
0      COVID_2020     hv1_var  rv1_var   144  4.259607  1823.975506
1      COVID_2020   ewma1_va

In [17]:
plot_vol_compare(df_har, "SPY", target_var="rv1_var", forecast_vars=("hv1_var","ewma1_var","garch1_var", "gjr1_var","har1_var"), crisis_windows = CRISIS_WINDOWS )

#### We can also try a simple ensemble as
$$ \sigma^2_{ensemble}  = 0.5 \times GJR + 0.5\times HAR $$

In [19]:
df_all = df_har.copy()
df_all["ens1_var"] = 0.5 * df_har["gjr1_var"] + 0.5 * df_har["har1_var"]
df_all["ens5_var"] = 5 * df_all["ens1_var"]
df_all.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,...,garch5_var,gjr1_var,gjr5_var,rv_d,rv_w,rv_m,har1_var,har5_var,ens1_var,ens5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.605412,4723500,-1.733110,3.003669,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,92.128937,5741700,0.342534,0.117329,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,22.071888,8405550,-2.388456,5.704724,...,NaN,NaN,NaN,5.704724,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,91.026451,7503700,-1.203895,1.449362,...,NaN,NaN,NaN,1.449362,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,22.210209,7271850,0.624728,0.390285,...,NaN,NaN,NaN,0.390285,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
models_1d = ["hv1_var", "ewma1_var", "garch1_var", "gjr1_var", "har1_var", "ens1_var"]
print(score(df_all, models_1d, "rv1_var", eval_start="2005-01-01"))
print()

models_5d = ["hv5_var", "ewma5_var", "garch5_var", "gjr5_var", "har5_var", "ens5_var"]
print(score(df_all, models_5d, "rv5_var", eval_start="2005-01-01"))
print()

print(score_by_regime(df_all, models_1d, "rv1_var", eval_start="2005-01-01"))

        model   target      n     QLIKE         MSE
3    gjr1_var  rv1_var  10652  1.359028  213.055874
5    ens1_var  rv1_var  10652  1.379218  214.001940
2  garch1_var  rv1_var  10652  1.393296  220.020207
4    har1_var  rv1_var  10652  1.446939  221.666354
1   ewma1_var  rv1_var  10652  1.450791  226.174898
0     hv1_var  rv1_var  10652  1.490452  235.490591

        model   target      n     QLIKE          MSE
3    gjr5_var  rv5_var  10652  2.831691   999.880640
2  garch5_var  rv5_var  10652  2.844596  1058.152795
5    ens5_var  rv5_var  10652  2.865038   826.642586
0     hv5_var  rv5_var  10652  2.890881  1456.071512
1   ewma5_var  rv5_var  10652  2.911794  1330.494781
4    har5_var  rv5_var  10652  2.936052   821.363259

           regime       model   target     n     QLIKE          MSE
3      COVID_2020    gjr1_var  rv1_var   144  3.724725  1546.460710
5      COVID_2020    ens1_var  rv1_var   144  3.748673  1477.062879
4      COVID_2020    har1_var  rv1_var   144  3.935383  146

### In general we can have
$$ \hat{\sigma}^2_{ensemble}  = w \hat{\sigma}^2_{GJR} + (1-w) \hat{\sigma}^2_{HAR} $$

where the weight $w$ can be determined through optimization e.g., minimizing QLIKE over $w$

In [35]:
def qlike_loss(y_true, y_pred, eps=1e-12):
    """
    Mean QLIKE loss for variance forecasts.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    y_true = np.maximum(y_true, eps)
    y_pred = np.maximum(y_pred, eps)

    return np.mean(np.log(y_pred) + y_true / y_pred)


def optimize_ensemble_weight(
    df,
    gjr_col="gjr1_var",
    har_col="har1_var",
    target_col="rv1_var",
    start="2005-01-01",
    end="2006-12-31",
    weight_grid=None
):
    """
    Find ensemble weight w that minimizes QLIKE on a development window.

    Ensemble:
        ens = w * GJR + (1-w) * HAR
    """
    if weight_grid is None:
        weight_grid = np.linspace(0.0, 1.0, 101)

    d = df.copy()
    d["date"] = pd.to_datetime(d["date"])

    d = d[
        (d["date"] >= pd.to_datetime(start)) &
        (d["date"] <= pd.to_datetime(end))
    ].copy()

    d = d.dropna(subset=[gjr_col, har_col, target_col]).copy()

    best_w = None
    best_loss = np.inf
    results = []

    for w in weight_grid:
        ens = w * d[gjr_col].values + (1.0 - w) * d[har_col].values
        loss = qlike_loss(d[target_col].values, ens)

        results.append({"w": w, "qlike": loss})

        if loss < best_loss:
            best_loss = loss
            best_w = w

    res_df = pd.DataFrame(results)
    return best_w, best_loss, res_df

def add_weighted_ensemble(
    df,
    w,
    gjr_col="gjr1_var",
    har_col="har1_var",
    out_col="ens1_var"
):
    """
    Add weighted ensemble forecast column:
        ens = w * GJR + (1-w) * HAR
    """
    out = df.copy()
    out[out_col] = w * out[gjr_col] + (1.0 - w) * out[har_col]
    return out

def add_weighted_ensemble_5d(
    df,
    w,
    gjr_col="gjr5_var",
    har_col="har5_var",
    out_col="ens5_var"
):
    out = df.copy()
    out[out_col] = w * out[gjr_col] + (1.0 - w) * out[har_col]
    return out

In [40]:
w_star, best_loss, w_table = optimize_ensemble_weight(
    df_har,
    gjr_col="gjr1_var",
    har_col="har1_var",
    target_col="rv1_var",
    start="2005-01-01",
    end="2006-12-31"
)

print("Optimal w:", w_star)
print("Best dev-window QLIKE:", best_loss)
w_table.head()

Optimal w: 1.0
Best dev-window QLIKE: 0.46388381481174223


,w,qlike
0,0.00,0.728949
1,0.01,0.726204
2,0.02,0.723452
3,0.03,0.720694
4,0.04,0.717930


In [26]:
df_ensemble = add_weighted_ensemble(
    df_har,
    w=w_star,
    gjr_col="gjr1_var",
    har_col="har1_var",
    out_col="ens1_var"
)

df_ensemble = add_weighted_ensemble_5d(
    df_ensemble,
    w=w_star,
    gjr_col="gjr5_var",
    har_col="har5_var",
    out_col="ens5_var"
)

In [28]:
models_1d = ["hv1_var", "ewma1_var", "garch1_var", "gjr1_var", "har1_var", "ens1_var"]
print(score(df_ensemble, models_1d, "rv1_var", eval_start="2007-01-01"))
print()

models_5d = ["hv5_var", "ewma5_var", "garch5_var", "gjr5_var", "har5_var", "ens5_var"]
print(score(df_ensemble, models_5d, "rv5_var", eval_start="2007-01-01"))
print()

print(score_by_regime(df_ensemble, models_1d, "rv1_var", eval_start="2007-01-01"))

        model   target     n     QLIKE         MSE
3    gjr1_var  rv1_var  9646  1.452385  235.055764
5    ens1_var  rv1_var  9646  1.452385  235.055764
2  garch1_var  rv1_var  9646  1.486818  242.739710
4    har1_var  rv1_var  9646  1.521820  244.412650
1   ewma1_var  rv1_var  9646  1.549412  249.537329
0     hv1_var  rv1_var  9646  1.589763  259.822764

        model   target     n     QLIKE          MSE
3    gjr5_var  rv5_var  9646  2.916891  1103.259421
5    ens5_var  rv5_var  9646  2.916891  1103.259421
2  garch5_var  rv5_var  9646  2.929656  1167.597433
0     hv5_var  rv5_var  9646  2.980784  1606.991010
4    har5_var  rv5_var  9646  3.002076   902.851673
1   ewma5_var  rv5_var  9646  3.003791  1468.306246

           regime       model   target     n     QLIKE          MSE
3      COVID_2020    gjr1_var  rv1_var   144  3.724725  1546.460710
5      COVID_2020    ens1_var  rv1_var   144  3.724725  1546.460710
4      COVID_2020    har1_var  rv1_var   144  3.935383  1460.542384
2    

In [34]:
import plotly.express as px

def plot_weight_search(w_table, title="Ensemble weight search"):
    fig = px.line(w_table, x="w", y="qlike", title=title)
    fig.update_layout(
        xaxis_title="Weight on GJR",
        yaxis_title="Development-window QLIKE"
    )
    fig.show()

plot_weight_search(w_table, title="GJR-HAR Ensemble Weight Search")